# Notebook 03 (Participant): Add a New Problem Scaffold

You will implement a robotics co-design problem contract and evaluate whether it is benchmark-ready.


**Edit-safe start:** this notebook opens from GitHub in read-only source mode. Use **File -> Save a copy in Drive** before running edits so your changes stay in your own workspace.


## Notebook map

This notebook is written as a standalone lab chapter:
- context first,
- implementation second,
- interpretation third.

If you are following asynchronously, run cells in order and use the success checks to validate each stage before moving on.


## Standalone guide

This chapter is about benchmark design quality, not model training speed.


## What makes a new problem benchmark-ready

A publishable benchmark needs explicit representation, constraints, objectives, simulator semantics, and reproducibility metadata.


In [ ]:
# Colab/local dependency bootstrap
import sys

IN_COLAB = 'google.colab' in sys.modules
FORCE_INSTALL = False  # Set True to force install outside Colab

if IN_COLAB or FORCE_INSTALL:
    print('Installing dependencies...')
    !pip install engibench[beams2d] matplotlib gymnasium pybullet
    try:
        import torch  # noqa: F401
    except Exception:
        !pip install torch torchvision
    print('Dependency install complete.')
else:
    print('Skipping install (using current environment). Set FORCE_INSTALL=True to install here.')


### Step 1 - Import scaffold dependencies

Ensure all required interfaces are visible before class implementation.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Annotated

import numpy as np
from gymnasium import spaces

from engibench.constraint import bounded
from engibench.constraint import constraint
from engibench.core import ObjectiveDirection
from engibench.core import OptiStep
from engibench.core import Problem

import pybullet as p


### Step 2 - Implement PyBullet manipulator co-design problem contract (TODO)

Complete each required method with deterministic behavior and clear failure messages.


In [ ]:
class PlanarManipulatorCoDesignProblem(Problem[np.ndarray]):
    """Robotics co-design scaffold using a real PyBullet rollout loop."""

    version = 0
    objectives = (
        ("final_tracking_error_m", ObjectiveDirection.MINIMIZE),
        ("actuation_energy_j", ObjectiveDirection.MINIMIZE),
    )

    @dataclass
    class Conditions:
        target_x: Annotated[float, bounded(lower=0.20, upper=1.35)] = 0.85
        target_y: Annotated[float, bounded(lower=0.05, upper=1.20)] = 0.45
        payload_kg: Annotated[float, bounded(lower=0.0, upper=2.0)] = 0.8
        disturbance_scale: Annotated[float, bounded(lower=0.0, upper=0.30)] = 0.05

    @dataclass
    class Config(Conditions):
        sim_steps: Annotated[int, bounded(lower=60, upper=1200)] = 240
        dt: Annotated[float, bounded(lower=1e-4, upper=0.05)] = 1.0 / 120.0
        torque_limit: Annotated[float, bounded(lower=1.0, upper=50.0)] = 12.0
        max_iter: Annotated[int, bounded(lower=1, upper=300)] = 60

    dataset_id = "IDEALLab/planar_manipulator_codesign_v0"  # placeholder for future dataset integration
    container_id = None

    def __init__(self, seed: int = 0, **kwargs):
        super().__init__(seed=seed)
        self.config = self.Config(**kwargs)
        self.conditions = self.Conditions(
            target_x=self.config.target_x,
            target_y=self.config.target_y,
            payload_kg=self.config.payload_kg,
            disturbance_scale=self.config.disturbance_scale,
        )

        # Design vector = [link1_m, link2_m, motor_strength, kp, kd, damping]
        self.design_space = spaces.Box(
            low=np.array([0.25, 0.20, 2.0, 5.0, 0.2, 0.0], dtype=np.float32),
            high=np.array([1.00, 0.95, 30.0, 120.0, 18.0, 1.5], dtype=np.float32),
            dtype=np.float32,
        )

        # TODO 1: implement design constraints and assign self.design_constraints
        # Suggested constraints:
        # - reachable workspace: link1+link2 must reach target radius
        # - gain consistency: kd must be bounded relative to kp
        raise NotImplementedError('Implement __init__ constraints')

    def _build_robot(self, l1: float, l2: float, payload_kg: float, damping: float) -> tuple[int, int]:
        # TODO 2: build 2-link planar robot in PyBullet DIRECT mode
        raise NotImplementedError('Implement _build_robot')

    def _inverse_kinematics_2link(self, x: float, y: float, l1: float, l2: float) -> tuple[float, float]:
        # TODO 3: implement closed-form IK for 2-link planar arm
        raise NotImplementedError('Implement _inverse_kinematics_2link')

    def _forward_kinematics_2link(self, q1: float, q2: float, l1: float, l2: float) -> tuple[float, float]:
        # TODO 4: implement FK for end-effector position
        raise NotImplementedError('Implement _forward_kinematics_2link')

    def _rollout(self, design: np.ndarray, cfg: dict, return_trace: bool = False):
        # TODO 5: run PyBullet rollout and compute objectives
        # Required outputs:
        # - final tracking error [m]
        # - actuation energy [J]
        # Optional trace for rendering: ee path, error curve, torque trace
        raise NotImplementedError('Implement _rollout')

    def simulate(self, design: np.ndarray, config: dict | None = None) -> np.ndarray:
        # TODO 6: call rollout with cfg merge and clipping to design bounds
        raise NotImplementedError('Implement simulate')

    def optimize(self, starting_point: np.ndarray, config: dict | None = None):
        # TODO 7: implement deterministic local search + OptiStep history
        raise NotImplementedError('Implement optimize')

    def render(self, design: np.ndarray, *, open_window: bool = False):
        # TODO 8: create 4-panel interpretation plot
        # Suggested panels: design vars, task-space path, error-vs-time, torque-vs-time
        raise NotImplementedError('Implement render')

    def random_design(self):
        # TODO 9: sample uniformly in design bounds
        raise NotImplementedError('Implement random_design')


### Step 3 - Smoke-test your scaffold

Run minimal checks to verify interface consistency and simulator behavior.


Use the multi-panel render to read **where heat enters**, **how material is distributed**, and **where thermal bottlenecks remain**.


Use the final figure to interpret whether the design/controller combination reaches the target robustly with acceptable energy use.


In [ ]:
# Run this cell after finishing TODOs in PlanarManipulatorCoDesignProblem
problem = PlanarManipulatorCoDesignProblem(
    seed=42,
    target_x=0.9,
    target_y=0.45,
    payload_kg=0.8,
    disturbance_scale=0.04,
    sim_steps=220,
    max_iter=40,
)
start, _ = problem.random_design()

cfg = {
    'target_x': 0.9,
    'target_y': 0.45,
    'payload_kg': 0.8,
    'disturbance_scale': 0.04,
    'sim_steps': 220,
    'dt': 1.0 / 120.0,
    'torque_limit': 12.0,
    'max_iter': 40,
}

print('design space:', problem.design_space)
print('objectives:', problem.objectives)
print('conditions:', problem.conditions)

viol = problem.check_constraints(start, config=cfg)
print('constraint violations:', len(viol))

obj0 = problem.simulate(start, config=cfg)
opt_design, history = problem.optimize(start, config=cfg)
objf = problem.simulate(opt_design, config=cfg)

print('initial objectives [tracking_error_m, energy_J]:', obj0.tolist())
print('final objectives   [tracking_error_m, energy_J]:', objf.tolist())
print('optimization steps:', len(history))
print('How to read plots: vars | task-space path | error timeline | torque timeline')

problem.render(opt_design)


## Mapping to real EngiBench contributions

Translate this robotics co-design scaffold into domain-specific simulators and datasets (robotics, controls, etc.) with documented assumptions.


## Contribution checklist

Before proposing a new problem, verify data provenance, split policy, evaluation protocol, and reporting templates.


## Troubleshooting

If a section fails, do not continue downstream. Fix locally first, then rerun the section and its immediate checks.
This notebook is intentionally staged so failures are localized.


## Takeaways

Before closing, record three points:
1. What conclusion is directly supported by your metrics?
2. What remains uncertain (and why)?
3. What extra experiment would you run next to reduce that uncertainty?


## Optional extension - Build a dataset and train a generative model

This section shows the full **offline benchmark loop** on the new problem:

1. Sample feasible designs and conditions with the simulator in-the-loop.
2. Build a compact training dataset.
3. Train a small conditional VAE (cVAE) to generate designs from conditions.
4. Evaluate generated designs against a random-design baseline.

Why this matters:
- It demonstrates that your scaffold is not just an API shell; it can support full generative benchmarking.
- It creates a reproducible path from simulator to learned design distribution.

Runtime note:
- Keep this optional (`RUN_OPTIONAL_SECTION=False`) during live sessions unless you have extra time.
- With default settings it should finish in a few minutes on Colab CPU/GPU.


In [ ]:
# Optional extension controls (safe defaults)
from pathlib import Path
import sys
import torch as th
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

RUN_OPTIONAL_SECTION = False  # Set True to run this optional extension
N_FEASIBLE_SAMPLES = 240
TOP_FRACTION = 0.35
EPOCHS = 18
BATCH_SIZE = 64
LATENT_DIM = 4
BETA_KL = 1e-3
FAST_SIM_CFG = {'sim_steps': 80, 'dt': 1.0 / 120.0}
EVAL_SAMPLES = 40

if 'problem' not in globals():
    problem = PlanarManipulatorCoDesignProblem(seed=7)

if 'google.colab' in sys.modules:
    OPTIONAL_ARTIFACT_DIR = Path('/content/dcc26_optional_artifacts')
else:
    OPTIONAL_ARTIFACT_DIR = Path('workshops/dcc26/optional_artifacts')
OPTIONAL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Optional artifacts dir: {OPTIONAL_ARTIFACT_DIR.resolve()}')
print('Optional section enabled:' if RUN_OPTIONAL_SECTION else 'Optional section disabled:', RUN_OPTIONAL_SECTION)


In [ ]:
# Build offline dataset from simulator rollouts
import numpy as np

rng = np.random.default_rng(123)


def sample_condition_dict() -> dict:
    return {
        'target_x': float(rng.uniform(0.20, 1.35)),
        'target_y': float(rng.uniform(0.05, 1.20)),
        'payload_kg': float(rng.uniform(0.0, 2.0)),
        'disturbance_scale': float(rng.uniform(0.0, 0.30)),
    }


def cond_to_vec(cfg: dict) -> np.ndarray:
    return np.array([
        cfg['target_x'],
        cfg['target_y'],
        cfg['payload_kg'],
        cfg['disturbance_scale'],
    ], dtype=np.float32)


def objective_score(obj: np.ndarray) -> float:
    # Same scalarization as quick optimization above: error + 0.02 * energy
    return float(obj[0] + 0.02 * obj[1])


def make_dataset(problem_obj, n_feasible: int):
    designs, conds, objs = [], [], []
    max_attempts = n_feasible * 6
    attempts = 0

    while len(designs) < n_feasible and attempts < max_attempts:
        attempts += 1
        d, _ = problem_obj.random_design()
        cfg = sample_condition_dict()

        violations = problem_obj.check_constraints(d, cfg)
        if len(violations) > 0:
            continue

        obj = problem_obj.simulate(d, {**cfg, **FAST_SIM_CFG})
        designs.append(d.astype(np.float32))
        conds.append(cond_to_vec(cfg))
        objs.append(obj.astype(np.float32))

        if len(designs) % 40 == 0:
            print(f'Collected feasible samples: {len(designs)}/{n_feasible}')

    if len(designs) < max(32, n_feasible // 3):
        raise RuntimeError(
            f'Not enough feasible samples ({len(designs)}). Increase attempts or relax settings.'
        )

    designs = np.stack(designs)
    conds = np.stack(conds)
    objs = np.stack(objs)
    scores = np.array([objective_score(o) for o in objs], dtype=np.float32)

    keep_n = max(32, int(TOP_FRACTION * len(scores)))
    top_idx = np.argsort(scores)[:keep_n]

    data = {
        'designs_all': designs,
        'conditions_all': conds,
        'objectives_all': objs,
        'scores_all': scores,
        'designs_top': designs[top_idx],
        'conditions_top': conds[top_idx],
        'objectives_top': objs[top_idx],
        'scores_top': scores[top_idx],
    }
    return data


if RUN_OPTIONAL_SECTION:
    dataset = make_dataset(problem, N_FEASIBLE_SAMPLES)
    np.savez(OPTIONAL_ARTIFACT_DIR / 'manipulator_dataset.npz', **dataset)
    print('Saved dataset:', OPTIONAL_ARTIFACT_DIR / 'manipulator_dataset.npz')
    print('All samples:', dataset['designs_all'].shape[0], '| Top samples:', dataset['designs_top'].shape[0])
else:
    dataset = None
    print('Skipped dataset creation. Set RUN_OPTIONAL_SECTION=True to run this block.')


### Optional model - Conditional VAE for inverse design

Modeling setup:
- **Condition input:** `(target_x, target_y, payload_kg, disturbance_scale)`
- **Generated output:** 6D design vector in the problem design space.
- **Training target:** top-performing feasible samples from the offline dataset.

This gives a minimal but valid generative baseline for `p(design | condition)`.


In [ ]:
# Define and train a small conditional VAE
class ConditionalVAE(nn.Module):
    def __init__(self, cond_dim: int, design_dim: int, latent_dim: int = 4, hidden: int = 96):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(cond_dim + design_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )
        self.mu = nn.Linear(hidden, latent_dim)
        self.logvar = nn.Linear(hidden, latent_dim)

        self.dec = nn.Sequential(
            nn.Linear(cond_dim + latent_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, design_dim),
            nn.Sigmoid(),
        )

    def encode(self, cond, design):
        h = self.enc(th.cat([cond, design], dim=-1))
        return self.mu(h), self.logvar(h)

    def reparameterize(self, mu, logvar):
        std = th.exp(0.5 * logvar)
        eps = th.randn_like(std)
        return mu + eps * std

    def decode(self, cond, z):
        return self.dec(th.cat([cond, z], dim=-1))

    def forward(self, cond, design):
        mu, logvar = self.encode(cond, design)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(cond, z)
        return recon, mu, logvar


if RUN_OPTIONAL_SECTION:
    lb = problem.design_space.low.astype(np.float32)
    ub = problem.design_space.high.astype(np.float32)

    x_cond = dataset['conditions_top'].astype(np.float32)
    y_design = dataset['designs_top'].astype(np.float32)
    y_norm = np.clip((y_design - lb) / (ub - lb + 1e-8), 0.0, 1.0)

    cond_t = th.tensor(x_cond, dtype=th.float32)
    design_t = th.tensor(y_norm, dtype=th.float32)
    loader = DataLoader(TensorDataset(cond_t, design_t), batch_size=BATCH_SIZE, shuffle=True)

    device = th.device('cuda' if th.cuda.is_available() else 'cpu')
    model = ConditionalVAE(cond_dim=4, design_dim=6, latent_dim=LATENT_DIM).to(device)
    opt = th.optim.Adam(model.parameters(), lr=2e-3)

    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        for c_batch, d_batch in loader:
            c_batch = c_batch.to(device)
            d_batch = d_batch.to(device)

            recon, mu, logvar = model(c_batch, d_batch)
            recon_loss = th.mean((recon - d_batch) ** 2)
            kl = -0.5 * th.mean(1 + logvar - mu.pow(2) - logvar.exp())
            loss = recon_loss + BETA_KL * kl

            opt.zero_grad()
            loss.backward()
            opt.step()
            epoch_loss += float(loss.item())

        epoch_loss /= max(1, len(loader))
        history.append(epoch_loss)
        if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
            print(f'Epoch {epoch:02d}/{EPOCHS} | loss={epoch_loss:.6f}')

    th.save(model.state_dict(), OPTIONAL_ARTIFACT_DIR / 'cvae_weights.pt')
    print('Saved model:', OPTIONAL_ARTIFACT_DIR / 'cvae_weights.pt')
else:
    model, history, device = None, [], th.device('cpu')
    print('Skipped model training. Set RUN_OPTIONAL_SECTION=True to run this block.')


In [ ]:
# Evaluate generated designs vs random baseline
import matplotlib.pyplot as plt


def sample_baseline(problem_obj, cfg: dict, trials: int = 8):
    best_obj = None
    for _ in range(trials):
        d, _ = problem_obj.random_design()
        if len(problem_obj.check_constraints(d, cfg)) > 0:
            continue
        obj = problem_obj.simulate(d, {**cfg, **FAST_SIM_CFG})
        if best_obj is None or objective_score(obj) < objective_score(best_obj):
            best_obj = obj
    if best_obj is None:
        # fallback if all random candidates violate constraints
        d, _ = problem_obj.random_design()
        best_obj = problem_obj.simulate(d, {**cfg, **FAST_SIM_CFG})
    return best_obj


def generate_design(model_obj, cfg_vec: np.ndarray, lb: np.ndarray, ub: np.ndarray):
    with th.no_grad():
        c = th.tensor(cfg_vec[None, :], dtype=th.float32, device=device)
        z = th.randn((1, LATENT_DIM), dtype=th.float32, device=device)
        d_norm = model_obj.decode(c, z).cpu().numpy()[0]
    d = lb + np.clip(d_norm, 0.0, 1.0) * (ub - lb)
    return d.astype(np.float32)


if RUN_OPTIONAL_SECTION:
    lb = problem.design_space.low.astype(np.float32)
    ub = problem.design_space.high.astype(np.float32)

    gen_objs = []
    base_objs = []
    feasible_count = 0

    for _ in range(EVAL_SAMPLES):
        cfg = sample_condition_dict()
        cfg_vec = cond_to_vec(cfg)

        # try a few generated candidates to satisfy constraints
        gen_obj = None
        for _retry in range(5):
            d_gen = generate_design(model, cfg_vec, lb, ub)
            if len(problem.check_constraints(d_gen, cfg)) == 0:
                gen_obj = problem.simulate(d_gen, {**cfg, **FAST_SIM_CFG})
                feasible_count += 1
                break
        if gen_obj is None:
            d_fallback, _ = problem.random_design()
            gen_obj = problem.simulate(d_fallback, {**cfg, **FAST_SIM_CFG})

        base_obj = sample_baseline(problem, cfg, trials=8)
        gen_objs.append(gen_obj)
        base_objs.append(base_obj)

    gen_objs = np.stack(gen_objs)
    base_objs = np.stack(base_objs)

    summary = {
        'generated_error_mean': float(np.mean(gen_objs[:, 0])),
        'generated_energy_mean': float(np.mean(gen_objs[:, 1])),
        'baseline_error_mean': float(np.mean(base_objs[:, 0])),
        'baseline_energy_mean': float(np.mean(base_objs[:, 1])),
        'generated_feasible_rate': float(feasible_count / EVAL_SAMPLES),
    }
    print('Optional extension summary:')
    for k, v in summary.items():
        print(f'  {k}: {v:.6f}')

    np.savez(OPTIONAL_ARTIFACT_DIR / 'optional_eval_summary.npz', gen_objs=gen_objs, base_objs=base_objs, **summary)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].plot(history)
    axes[0].set_title('cVAE training loss')
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('loss')
    axes[0].grid(alpha=0.3)

    axes[1].hist(base_objs[:, 0], bins=12, alpha=0.6, label='baseline')
    axes[1].hist(gen_objs[:, 0], bins=12, alpha=0.6, label='generated')
    axes[1].set_title('Final tracking error')
    axes[1].set_xlabel('error [m]')
    axes[1].legend()

    axes[2].hist(base_objs[:, 1], bins=12, alpha=0.6, label='baseline')
    axes[2].hist(gen_objs[:, 1], bins=12, alpha=0.6, label='generated')
    axes[2].set_title('Actuation energy')
    axes[2].set_xlabel('energy [J]')
    axes[2].legend()

    fig.tight_layout()
    plt.show()
else:
    print('Skipped evaluation. Set RUN_OPTIONAL_SECTION=True to run this block.')


### Discussion prompts for workshop synthesis

Use this optional experiment to trigger discussion:

1. Is simulator-generated offline data enough for credible inverse-design benchmarking?
2. Which metrics should become mandatory in cross-domain comparisons (quality, feasibility, diversity, cost)?
3. How would you package this scaffold into a reusable EngiBench-style contribution (dataset card, baselines, splits, eval protocol)?
